# Lab02 — Example: các bước trên một ma trận

(Cập nhật lần cuối 25/09/2026)

File này **không tính điểm và không cần nộp**. Nó minh hoạ trên dữ liệu rất nhỏ những việc bạn sẽ làm trên dữ liệu thật ở `Lab02.ipynb`:
1. Công thức độ tương tự và trọng số (giữ nguyên từ Example của năm trước).
2. Cách **đo** thời gian và bộ nhớ đỉnh.
3. Ví dụ số của đẳng thức ngưỡng dùng ở phần B3.
4. Một hành vi của `np.nansum` bạn cần biết.

In [ ]:
import time
import tracemalloc
import numpy as np

---

## 1. Độ tương tự và trọng số trên ma trận 4×5

Giả sử ta có ma trận `ratings`, trong đó mỗi phần tử là điểm mà một người dùng (dòng) chấm cho một bộ phim (cột); `NaN` là chưa chấm.

In [ ]:
ratings = np.array([[5, 4, np.nan, 1, np.nan], [np.nan, 4, 4, 2, 2],
                    [3, np.nan, np.nan, 5, 5],[np.nan, 3, 4, 4, np.nan]])
ratings

array([[ 5.,  4., nan,  1., nan],
       [nan,  4.,  4.,  2.,  2.],
       [ 3., nan, nan,  5.,  5.],
       [nan,  3.,  4.,  4., nan]])

In [ ]:
np.set_printoptions(suppress = True)

Ở đây mình chỉ tính cho **một** người dùng; phần tính cho cả batch (nhiều người dùng) bạn sẽ tự làm trong bài. Ta sẽ dự đoán điểm cho các phim người dùng này chưa chấm, tức là các ô `NaN`. Đầu tiên, ta xem mỗi dòng là một vector ứng với một người dùng, rồi tính độ tương tự giữa các vector thông qua khoảng cách.

In [ ]:
# Lấy ra user đầu tiên
r0 = ratings[0]

# Lấy tất cả các user trừ cho user đầu tiên để tính khoảng cách,
# sau đó tính mean của từng dòng để thu được độ tương tự giữa các user đối với user[0]
s = np.nanmean(np.abs(ratings - r0), axis = 1)

s = 1/(s + 0.001)
s

array([1000.        ,    1.99600798,    0.33322226,    0.49975012])

Để ý phần tử đầu tiên: độ tương tự của một người dùng với chính mình là `1/0.001 = 1000`.

In [ ]:
# Lấy ra các cột mà user[0] chưa đánh giá
a = ratings[:, np.isnan(r0)]
a

array([[nan, nan],
       [ 4.,  2.],
       [nan,  5.],
       [ 4., nan]])

In [ ]:
# Tạo ma trận trọng số dựa trên độ tương tự
w = ~np.isnan(a)*s.reshape(-1, 1)
# Chuẩn hoá trọng số
w = w/w.sum(axis = 0)
w

array([[0.        , 0.        ],
       [0.79976019, 0.85693889],
       [0.        , 0.14306111],
       [0.20023981, 0.        ]])

In [ ]:
# Kết quả dự đoán được
np.nansum(a*w, axis = 0)

array([4.        , 2.42918332])

Trong bài, bạn làm đúng các bước này nhưng cho **cả một batch** người dùng cùng lúc, nên mảng trọng số có thêm một chiều: `(batch_size, n_users, n_movies)`.

---

## 2. Đo thay vì đoán

Hai công cụ dùng xuyên suốt bài:
- `time.perf_counter()` — đồng hồ chính xác nhất của Python để đo **thời gian**.
- `tracemalloc` — theo dõi **bộ nhớ đỉnh** (peak) mà Python và NumPy cấp phát trong một đoạn code. NumPy báo mọi lần cấp phát mảng cho `tracemalloc`, nên nó đo được cả những mảng trung gian mà bạn không gán vào biến nào.

Ví dụ dưới đây đo phép trừ broadcasting `X[:, None, :] - X[None, :, :]` trên ma trận ngẫu nhiên `X` kích thước `n × 200` với vài giá trị `n`, rồi so bộ nhớ đỉnh đo được với con số **tính bằng giấy**: mảng kết quả có `n · n · 200` phần tử `float64`, mỗi phần tử 8 byte.

In [ ]:
rng = np.random.RandomState(0)
for n in [50, 100, 200, 400]:
    X = rng.rand(n, 200)
    tracemalloc.start()
    t0 = time.perf_counter()
    D = np.abs(X[:, None, :] - X[None, :, :])
    elapsed = time.perf_counter() - t0
    peak = tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    on_paper = n * n * 200 * 8
    print(f'n = {n:3d}: {elapsed * 1000:7.1f} ms, '
          f'peak đo được {peak / 2**20:6.1f} MiB, '
          f'tính bằng giấy {on_paper / 2**20:6.1f} MiB')
    del D

n =  50:     6.4 ms, peak đo được    7.6 MiB, tính bằng giấy    3.8 MiB
n = 100:    15.5 ms, peak đo được   30.5 MiB, tính bằng giấy   15.3 MiB
n = 200:    60.4 ms, peak đo được  122.1 MiB, tính bằng giấy   61.0 MiB


n = 400:   226.1 ms, peak đo được  488.3 MiB, tính bằng giấy  244.1 MiB


Hai điều cần để ý:
- `n` tăng gấp đôi thì bộ nhớ tăng **gấp bốn**. Mảng trung gian của broadcasting lớn theo **tích** các chiều, nên rất nhanh vượt quá RAM.
- Bộ nhớ đỉnh đo được lớn hơn con số tính bằng giấy: `np.abs(A - B)` tạo mảng `A - B` trước, rồi mới tạo mảng kết quả của `abs`, nên có lúc hai mảng cùng tồn tại.

Cách dùng `tracemalloc`: `start()` → chạy code → đọc `get_traced_memory()[1]` (phần tử `[1]` là đỉnh) → `stop()`. Nhớ `stop()` sau mỗi lần đo, nếu không lần đo sau sẽ cộng dồn.

---

## 3. Đẳng thức ngưỡng dùng ở B3

Với hai số nguyên $a, b$ trong khoảng 1–5:
$$|a - b| = \sum_{t=1}^{4} \big([a > t] \ \text{xor}\ [b > t]\big)$$

Ví dụ $a = 4$, $b = 2$:

| $t$ | $[4 > t]$ | $[2 > t]$ | xor |
|---|---|---|---|
| 1 | 1 | 1 | 0 |
| 2 | 1 | 0 | 1 |
| 3 | 1 | 0 | 1 |
| 4 | 0 | 0 | 0 |
| **tổng** | | | **2** = $\lvert 4 - 2 \rvert$ |

Ý nghĩa: mỗi ngưỡng $t$ nằm **giữa** $a$ và $b$ đóng góp đúng 1, và có đúng $|a-b|$ ngưỡng như vậy. Kiểm cho mọi cặp $(a, b)$:

In [ ]:
ok = True
for a in range(1, 6):
    for b in range(1, 6):
        total = sum((a > t) != (b > t) for t in range(1, 5))
        ok = ok and (total == abs(a - b))
print('Đẳng thức đúng với mọi cặp (a, b) trong 1..5:', ok)

Đẳng thức đúng với mọi cặp (a, b) trong 1..5: True


Với hai số 0/1 $x, y$ thì $x \ \text{xor}\ y = x + y - 2xy$ (bạn thử cả 4 trường hợp). Nhờ vậy, một **tổng** các phép xor viết được bằng phép cộng và phép nhân, là những thứ mà phép nhân ma trận làm rất giỏi.

Một lưu ý nhỏ khi so sánh với ngưỡng trên mảng có `NaN`: `np.nan > t` luôn là `False`.

In [ ]:
v = np.array([5, np.nan, 2])
v > 3

array([ True, False, False])

---

## 4. `np.nansum` trên một dãy toàn `NaN`

`np.nansum` bỏ qua các `NaN` rồi cộng phần còn lại. Nếu **mọi** phần tử đều là `NaN` thì "phần còn lại" rỗng, và tổng của một dãy rỗng là 0. Trong khi đó `np.nanmean` của dãy toàn `NaN` trả về `NaN` (kèm một cảnh báo).

In [ ]:
import warnings
x = np.array([np.nan, np.nan, np.nan])
with warnings.catch_warnings():
    warnings.simplefilter('ignore', RuntimeWarning)
    print('np.nansum :', np.nansum(x))
    print('np.nanmean:', np.nanmean(x))

np.nansum : 0.0
np.nanmean: nan


Bạn nhớ hành vi này khi làm câu hỏi cuối phần B của bài.